In [ ]:
from __future__ import annotations

import os

import numpy as np
import uproot


def run_command(command, dry_run=False):
    print(command)
    if not dry_run:
        os.system(command)

In [ ]:
ntoys = 40

# run_command(f"python3 TestToys.py  --method abcd --tag test --ntoys {ntoys}")

In [ ]:
for itoy in range(16, ntoys):
    templates_dir = (
        f"/home/users/cmantill/hh/HH4b/src/HH4b/boosted/templates/alltoys_test/toy_{itoy}"
    )
    year = "2022EE"
    model_name = f"toy_{itoy}"
    command = f"python3 ../postprocessing/CreateDatacard.py --templates-dir {templates_dir} --year {year} --model-name {model_name} --cards-dir cards --regions pass_bin1 --unblinded"
    run_command(command)

In [ ]:
r_bounds = [-50, 50]

for itoy in range(16, ntoys):
    command = f"cd cards/toy_{itoy}; "
    command += "combineCards.py fail=fail.txt passbin1=passbin1.txt > combined.txt; "
    command += "text2workspace.py combined.txt; "
    command += f"combine -M FitDiagnostics --trackParameters r --trackErrors r --justFit -m 125 -d combined.root --rMin {r_bounds[0]} --rMax {r_bounds[1]} --robustFit=1 --X-rtd MINIMIZER_MaxCalls=1000000 --cminDefaultMinimizerTolerance 0.1"
    run_command(command)

In [ ]:
r_dict = {}
biases = [0]
for bias in biases:
    file_names = "/home/users/woodson/HH4b/src/HH4b/boosted/cards/toy_*/higgsCombineTest.FitDiagnostics.mH125.root"
    file = uproot.concatenate(file_names)

    r = np.array(file.limit)[::4]
    neg_lim = np.array(file.limit)[1::4]
    pos_lim = np.array(file.limit)[2::4]
    r_negerr = r - neg_lim
    r_poserr = pos_lim - r
    reldiff = r - bias
    reldiff[reldiff < 0] = (reldiff / r_poserr)[reldiff < 0]
    reldiff[reldiff > 0] = (reldiff / r_negerr)[reldiff > 0]

    r_dict[bias] = {
        "r": r,
        "reldiff": reldiff,
        "neg_lim": neg_lim,
        "pos_lim": pos_lim,
    }

In [ ]:
import matplotlib.pyplot as plt
import mplhep as hep
from scipy import stats

xrange = 4
bins = 21
x = np.linspace(-xrange, xrange, 101)


fig, axs = plt.subplots(len(biases), 1, figsize=(12, len(biases) * 10))

for i, bias in enumerate(biases):
    r_lims_bounds = (
        (r_dict[bias]["reldiff"] < 0) * (np.isclose(r_dict[bias]["pos_lim"], r_bounds[1]))
    ) + ((r_dict[bias]["reldiff"] > 0) * (np.isclose(r_dict[bias]["neg_lim"], r_bounds[0])))

    r_lims_same = r_dict[bias]["pos_lim"] == r_dict[bias]["neg_lim"]

    fit_fail = r_lims_bounds + r_lims_same

    r = r_dict[bias]["r"][~fit_fail]
    reldiff = r_dict[bias]["reldiff"][~fit_fail]
    reldiff = reldiff[(reldiff > -xrange) * (reldiff < xrange)]

    mu, sigma = np.mean(reldiff), np.std(reldiff)

    ax = axs[i] if len(biases) > 1 else axs

    ax.hist(
        reldiff,
        np.linspace(-xrange, xrange, bins + 1),
        histtype="step",
        label=f"{len(reldiff)} Toys",
    )
    ax.plot(
        x,
        # scale by bin width
        stats.norm.pdf(x, loc=mu, scale=sigma) * len(r) * (2 * xrange / bins),
        label=rf"$\mu = {mu:.2f}, \sigma = {sigma:.2f}$",
    )
    ax.set_xlabel(rf"$\frac{{\hat{{r}} - {bias}}}{{\Delta \hat r}}$")
    ax.set_ylabel("Number of toys")
    ax.set_title(f"r = {bias}")
    ax.legend(title="Combined")

    hep.cms.label(
        "Preliminary",
        ax=ax,
        data=True,
        lumi=61,
        year=None,
    )

plt.savefig("bias.pdf", bbox_inches="tight")
plt.show()